# 02 — Missing values: Chennai & Hyderabad (Google Colab)
Làm theo quy trình của `01_clean_delhi_gurugram.ipynb`, dùng chung `src/utils.py` của repo.
Bước **Outliers** để làm sau.

**Quy ước của nhóm:** chuỗi 0 liên tiếp ≥ 3 ngày → NaN · chỉ nội suy khoảng thiếu ≤ 14 ngày · khối trống dài giữ NaN · không tự điền `AQI`.

## 0. Lấy code và dữ liệu từ GitHub

In [ ]:
import os, sys

REPO = "/content/Air-Project"
if not os.path.exists(REPO):
    !git clone https://github.com/MTuan1510/Air-Project.git {REPO}
else:
    !git -C {REPO} pull

sys.path.append(f"{REPO}/src")
RAW = f"{REPO}/data/raw/city_day.csv"
INTERIM = f"{REPO}/data/interim"
os.makedirs(INTERIM, exist_ok=True)
print("Có file dữ liệu:", os.path.exists(RAW))

In [ ]:
import pandas as pd
import numpy as np
from utils import audit, load_and_split_city, diagnose_missing, impute_time_series

POLL = ['PM2.5','PM10','NO','NO2','NOx','NH3','CO','SO2','O3','Benzene','Toluene','Xylene']

chennai = load_and_split_city("Chennai", raw_path=RAW)
hyderabad = load_and_split_city("Hyderabad", raw_path=RAW)

print("=== CHENNAI (audit gốc) ===")
display(audit(chennai).sort_values("pct_missing", ascending=False))
print("\n=== HYDERABAD (audit gốc) ===")
display(audit(hyderabad).sort_values("pct_missing", ascending=False))

## 1. Chẩn đoán missing

In [ ]:
print("=== CHENNAI - diagnose_missing ===")
display(diagnose_missing(chennai))
print("\n=== HYDERABAD - diagnose_missing ===")
display(diagnose_missing(hyderabad))

**Quyết định drop (cả nhóm cần đồng ý):**
- Chennai `Xylene`: thiếu 100% → drop.
- Chennai `PM10`: thiếu 85%, chỉ có dữ liệu từ 2019 → chưa đạt ngưỡng drop 90%, giữ NaN như PM10 của Gurugram.
- Hyderabad: không drop cột nào.

## 2. Xử lý missing

In [ ]:
chennai_clean, chennai_flat = impute_time_series(chennai, "Chennai", drop_cols=["Xylene"])
print()
hyderabad_clean, hyderabad_flat = impute_time_series(hyderabad, "Hyderabad", drop_cols=[])

## 3. Kiểm chứng trước / sau

In [ ]:
def compare_missing(before_df, after_df, cols):
    before = before_df[cols].isna().mean() * 100
    after_cols = [c for c in cols if c in after_df.columns]
    after = after_df[after_cols].isna().mean() * 100
    return pd.DataFrame({'truoc_%': before.round(1), 'sau_%': after.reindex(before.index).round(1)})

print("=== CHENNAI ===");   display(compare_missing(chennai, chennai_clean, POLL + ['AQI']))
print("\n=== HYDERABAD ==="); display(compare_missing(hyderabad, hyderabad_clean, POLL + ['AQI']))

## 4. Lưu và tải về máy

In [ ]:
chennai_clean.to_csv(f"{INTERIM}/chennai_after_missing.csv", index=False)
hyderabad_clean.to_csv(f"{INTERIM}/hyderabad_after_missing.csv", index=False)
print("Chennai:", chennai_clean.shape, "| Hyderabad:", hyderabad_clean.shape)

from google.colab import files
files.download(f"{INTERIM}/chennai_after_missing.csv")
files.download(f"{INTERIM}/hyderabad_after_missing.csv")